# Simulating some NR events

Pueh Leng Tan, 10 March 2026

In [1]:
import os
from time import time

import numpy as np
import pandas as pd
import scipy.stats as sps
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import jax.numpy as jnp
import multihist as mh

import appletree as apt
from appletree.utils import get_file_path

XLA_PYTHON_CLIENT_PREALLOCATE is set to false
XLA_PYTHON_CLIENT_ALLOCATOR is set to platform


/home/puehlengt/.local/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


Using aptext package from https://github.com/XENONnT/applefiles


In [2]:
# constrain the GPU memory usage
apt.set_gpu_memory_usage(0.5)

## Define component

### ComponentSim

In [ ]:
# The components is associated with bins, so first we load bins
data = pd.read_csv(get_file_path("data_Rn220.csv"))
bins_cs1, bins_cs2 = apt.utils.get_equiprob_bins_2d(
    data[["cs1", "cs2"]].to_numpy(),
    [15, 15],
    order=[0, 1],
    x_clip=[0, 100],
    y_clip=[1e2, 1e4],
    which_np=jnp,
)

In [ ]:
# Initialize component
#nr = apt.NR(bins=[bins_cs1, bins_cs2], bins_type="irreg")
nr = apt.NR()

In [ ]:
# Deduce the workflow(datastructure)
nr.deduce(data_names=["cs1", "cs2"], func_name="simulate")  # 'eff'(efficiency) is always simulated
nr.rate_name = "nr_rate"  # also we have to specify a normalization factor of the component

# Compile ER script
# This is meta-programing because  appletree can generate codes dynamically
nr.compile()

In [ ]:
# For reference, this is the compiled code, the function is stored in appletree.share._cached_functions
# Initialize component
print('NR')
print(nr.code)


## Load parameters

In [ ]:
# Of course we have to load parameters (and their priors) in simulation (who the hell writes such comments..)
par_manager = apt.Parameter(get_file_path("nr_low.json"))

## Simulation

In [ ]:
num_sigmas = 6
tail_prob = sps.norm.sf(x=num_sigmas)
suggested_max_batch = sps.norm.ppf(1-tail_prob,
                                   loc=par_manager.par_config['nr_rate']['init_mean'],
                                   scale=par_manager.par_config['nr_rate']['init_mean'])
print(suggested_max_batch) # number of NR events hardly gonna fluctuate above this

In [ ]:
batch_size = int(14e3)
num_sims = int(3000)

param_bag = []
events_bag = []

for _mc in range(num_sims):
    #print(_mc)
    key = apt.randgen.get_key(seed=_mc)
    par_manager.sample_prior() # sampling from prior
    parameters = par_manager.get_all_parameter()

    key, tmp = nr.simulate(key, batch_size, parameters)
    events = np.asarray(jnp.stack(tmp, axis=1))   # shape (n, 3)
    #print(events)
    param_bag.append(parameters.copy())
    events_bag.append(events)

In [ ]:
target = 200_000

24.6/3000*target/60. # mins

In [ ]:
len(events_bag)

In [ ]:
_mc = 2

this_params = param_bag[_mc]
this_events = events_bag[_mc][:, :2]
this_eff = events_bag[_mc][:, -1].astype(bool)

In [ ]:
this_params, len(this_params)

In [ ]:
plt.plot(this_events[:, 0], this_events[:, 1], '.', label='all events')
plt.plot(this_events[:, 0][~this_eff], this_events[:, 1][~this_eff], 'rx', label='eff = 0')

#plt.yscale('log')
plt.xlabel('cs1 [PE]')
plt.ylabel('cs2 [PE]')
plt.legend()

In [ ]:
# todo:
# 1. save the simulated data
# 2. visualize the simulated data
# 3. think of a way to sample number of events according to the rate parameter